In [35]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import fcluster

In [91]:
master = pd.read_parquet(
    "../data/master/master_dwei.parquet"
)

master.shape

(632, 62)

In [93]:
features_A = [
    "DWEI_score",

    "female_literacy_pct",
    "scst_pct",
    "agri_worker_pct",
    "poverty_log",
    "night_lights_log",

    "wage_timeliness_pct",
    "avg_days_per_hh",
    "women_pct",
    "persondays_per_hh"
]
#Efficiency considering both
#structural conditions and implementation.

In [94]:
cluster_features = [
    "DWEI_score",
    "wage_timeliness_pct",
    "avg_days_per_hh",
    "women_pct",
    "persondays_per_hh"
]

In [95]:
cluster_df = (
    master[
        ["State", "District"] + cluster_features
    ]
    .dropna()
    .copy()
)

In [96]:
scaler = StandardScaler()

X = scaler.fit_transform(
    cluster_df[cluster_features]
)

In [97]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=20
)

cluster_df["cluster"] = (
    kmeans.fit_predict(X)
)

In [98]:
cluster_df.groupby("cluster")[
    cluster_features
].mean().round(3)

,DWEI_score,wage_timeliness_pct,avg_days_per_hh,women_pct,persondays_per_hh
cluster,,,,,
0,-0.417,98.723,34.240,46.423,34.735
1,-0.117,99.362,29.330,77.321,29.820
2,0.198,97.858,27.308,40.575,27.804
3,-0.340,35.111,20.237,30.209,20.620
4,0.279,99.736,43.081,46.785,43.579


In [99]:
cluster_order = (
    cluster_df
    .groupby("cluster")["DWEI_score"]
    .mean()
    .sort_values()
)

cluster_order

cluster
0   -0.417053
3   -0.340477
1   -0.117301
2    0.198236
4    0.279185
Name: DWEI_score, dtype: float64

In [100]:
cluster_to_tier = {
    4: "Tier I - High Impact Districts",
    2: "Tier II - Strong Performing Districts",
    1: "Tier III - Inclusive Development Districts",
    0: "Tier IV - Improvement Potential Districts",
    3: "Tier V - Special Challenge Districts"
}

In [101]:
cluster_df["tier"] = (
    cluster_df["cluster"]
    .map(cluster_to_tier)
)

In [102]:
cluster_df["tier"].value_counts()

tier
Tier II - Strong Performing Districts         198
Tier IV - Improvement Potential Districts     159
Tier I - High Impact Districts                157
Tier III - Inclusive Development Districts     92
Tier V - Special Challenge Districts           17
Name: count, dtype: int64

In [103]:
master = master.merge(
    cluster_df[
        [
            "State",
            "District",
            "cluster",
            "tier"
        ]
    ],
    on=["State", "District"],
    how="left"
)

In [104]:
master[
    [
        "State",
        "District",
        "DWEI_score",
        "cluster",
        "tier"
    ]
].head()

,State,District,DWEI_score,cluster,tier
0,Andaman And Nicobar Islands,Nicobars,-0.572450,3.0,Tier V - Special Challenge Districts
1,Andaman And Nicobar Islands,North And Middle Andaman,0.758327,2.0,Tier II - Strong Performing Districts
2,Andaman And Nicobar Islands,South Andamans,0.158898,2.0,Tier II - Strong Performing Districts
3,Andhra Pradesh,Ananthapuramu,-0.357821,4.0,Tier I - High Impact Districts
4,Andhra Pradesh,Chittoor,0.116049,4.0,Tier I - High Impact Districts


In [105]:
master[
    [
        "cluster",
        "tier"
    ]
].isna().sum()

cluster    9
tier       9
dtype: int64

In [106]:
master.groupby(
    "tier"
).agg(
    districts=("District", "count"),
    avg_dwei=("DWEI_score", "mean")
).round(3)

,districts,avg_dwei
tier,,
Tier I - High Impact Districts,157,0.279
Tier II - Strong Performing Districts,198,0.198
Tier III - Inclusive Development Districts,92,-0.117
Tier IV - Improvement Potential Districts,159,-0.417
Tier V - Special Challenge Districts,17,-0.340


In [107]:
master.to_parquet(
    "../data/master/master_clustered.parquet",
    index=False
)

master.to_csv(
    "../data/master/master_clustered.csv",
    index=False
)

In [108]:
import joblib

joblib.dump(
    kmeans,
    "../models/kmeans_tiers.pkl"
)

['../models/kmeans_tiers.pkl']

In [115]:
Z = linkage(
    X,
    method="ward"
)

In [120]:
cluster_df["hc_cluster"] = fcluster(
    Z,
    t=5,
    criterion="maxclust"
)

In [121]:
cluster_df["hc_cluster"].value_counts()

hc_cluster
5    210
3    166
1    124
2    109
4     14
Name: count, dtype: int64

In [122]:
from sklearn.metrics import adjusted_rand_score

In [123]:
ari = adjusted_rand_score(
    cluster_df["cluster"],
    cluster_df["hc_cluster"]
)

print(ari)

0.4102523293798912


In [124]:
pd.crosstab(
    cluster_df["cluster"],
    cluster_df["hc_cluster"]
)

hc_cluster,1,2,3,4,5
cluster,,,,,
0,44,84,1,0,30
1,63,0,0,0,29
2,0,14,36,0,148
3,0,0,0,14,3
4,17,11,129,0,0


Hierarchical clustering was used as a robustness check for the KMeans-based district typology. The Adjusted Rand Index (ARI) between the two methods was 0.41, indicating moderate agreement. This suggests that district governance patterns form a continuum rather than sharply separated groups. Consequently, the KMeans solution was retained for interpretability and policy communication purposes.